# `main.ipynb`: main simulation notebook
## Initialization
Load packages

In [34]:
import jax
from jax import numpy as jnp
from jax import random
from pathlib import Path
from tqdm import tqdm

from utils.criteria import *
from utils.energies import *
from utils.mcmc import *
from utils.modelIO import *
from utils.phases import *
from utils.stability import *

Load data and initialize

In [35]:
GRN_PATH = Path('data/eml/eml_fate_network.npy')
GIDS_PATH = Path('data/eml/eml_gene_ids.csv')
SAVE_PATH = Path('results/eml/')
RGRNS_PATH = SAVE_PATH.joinpath('random_networks.npy')
SEED = 1234
bio_grn, gids, N = load_grn(GRN_PATH, GIDS_PATH)
criterion = GlauberCriterion()

coarse_temperatures = jnp.linspace(0.0, 5.0, 11)
fine_temperatures = jnp.linspace(1.0, 2.5, 151)
varying_fields = jnp.arange(-6, 7, 1)
zero_field = jnp.zeros(N)
nonzero_field = jnp.zeros(N).at[4].set(1.0).at[9].set(-1.0)

key = random.key(SEED)
keys_4 = random.split(key, num=4) 
rgrn_keys = random.split(keys_4[0], num=100000)
keys_1e2 = random.split(keys_4[1], num=100)
keys_1e3 = random.split(keys_4[2], num=1000)
keys_1e5 = random.split(keys_4[3], num=100000)

Get random networks

In [ ]:
def rand_pair_sim_rgrn(key):
    return randomize_grn(key, bio_grn) # get random network
rand_pair_sim_rgrn_optimized = jax.jit(jax.vmap(rand_pair_sim_rgrn))

rgrns = rand_pair_sim_rgrn_optimized(rgrn_keys)
jnp.save(RGRNS_PATH, rgrns)

Prime functions for trajectories

In [17]:
# prime function for biological network with random initial conditions
def bio_ric_sim_prime(key, T, field, N_steps):
    grn_state = random.randint(key, (), 0, 2**N) # initialize replica state
    _, key = random.split(key) # refresh key
    return get_trajectory(key, grn_state, bio_grn, field, T, criterion, N, N_steps)

# prime function for biological network with specific initial conditions
def bio_spec_sim_prime(grn_state, key, T, field, N_steps):
    return get_trajectory(key, grn_state, bio_grn, field, T, criterion, N, N_steps)

# prime function for random networks with random initial conditions
def rand_pair_sim_prime(key_rgrn_pair, T, field, N_steps):
    key, rgrn = key_rgrn_pair # unpack key and random network
    grn_state = random.randint(key, (), 0, 2**N) # initialize replica state
    _, key = random.split(key) # refresh key
    return get_trajectory(key, grn_state, rgrn, field, T, criterion, N, N_steps)

Prime functions for final states

In [18]:
# prime function for biological network with random initial conditions
def bio_ric_sim_prime_fs(key, T, field, N_steps):
    grn_state = random.randint(key, (), 0, 2**N) # initialize replica state
    _, key = random.split(key) # refresh key
    return get_final_state(key, grn_state, bio_grn, field, T, criterion, N, N_steps)

# prime function for biological network with specific initial conditions
def bio_spec_sim_prime_fs(grn_state, key, T, field, N_steps):
    return get_final_state(key, grn_state, bio_grn, field, T, criterion, N, N_steps)

# prime function for random networks with random initial conditions
def rand_pair_sim_prime_fs(key_rgrn_pair, T, field, N_steps):
    key, rgrn = key_rgrn_pair # unpack key and random network
    grn_state = random.randint(key, (), 0, 2**N) # initialize replica state
    _, key = random.split(key) # refresh key
    return get_final_state(key, grn_state, rgrn, field, T, criterion, N, N_steps)

## Coarsely-spaced temperatures

Biological GRN: 1000 replicas x coarsely spaced temperatures

In [5]:
bio_T_coarse_path = SAVE_PATH.joinpath('bio_T_coarse')
bio_T_coarse_path.mkdir(parents=True, exist_ok=True)

for T in tqdm(coarse_temperatures): # for loop over temperatures due to memory limit
    xs_filename = bio_T_coarse_path.joinpath('bio_cT_{:02}_xs.npy'.format((10*T).astype(int)))
    vs_filename = bio_T_coarse_path.joinpath('bio_cT_{:02}_vs.npy'.format((10*T).astype(int)))

    def bio_sim_T_vary(key): # fix h=0, N_steps=N**2
        return bio_ric_sim_prime(key, T, zero_field, N**2)
    bio_sim_T_vary_optimized = jax.jit(jax.vmap(bio_sim_T_vary))

    xs, vs = bio_sim_T_vary_optimized(keys_1e3)
    jnp.save(xs_filename, xs)
    jnp.save(vs_filename, vs)

100%|██████████| 11/11 [00:13<00:00,  1.19s/it]


Random GRNs: 1000 GRN-replica pairs x coarsely spaced temperatures

In [6]:
rand_T_coarse_path = SAVE_PATH.joinpath('rand_T_coarse')
rand_T_coarse_path.mkdir(parents=True, exist_ok=True)
rgrns_1e3 = jnp.load(RGRNS_PATH)[:1000]

for T in tqdm(coarse_temperatures): # for loop over temperatures due to memory limit
    xs_filename = rand_T_coarse_path.joinpath('rand_cT_{:02}_xs.npy'.format((10*T).astype(int)))
    vs_filename = rand_T_coarse_path.joinpath('rand_cT_{:02}_vs.npy'.format((10*T).astype(int)))

    def rand_pair_sim_T_vary(rand_pair): # fix h=0, N_steps=N**2
        return rand_pair_sim_prime(rand_pair, T, zero_field, N**2)
    rand_sim_T_vary_optimized = jax.jit(jax.vmap(rand_pair_sim_T_vary))

    xs, vs = rand_sim_T_vary_optimized((keys_1e3, rgrns_1e3))
    jnp.save(xs_filename, xs)
    jnp.save(vs_filename, vs)

100%|██████████| 11/11 [00:13<00:00,  1.19s/it]


## Finely-spaced temperatures

Biological GRN: 1000 replicas x finely spaced temperatures

In [4]:
bio_T_fine_path = SAVE_PATH.joinpath('bio_T_fine')
bio_T_fine_path.mkdir(parents=True, exist_ok=True)

for T in tqdm(fine_temperatures): # for loop over temperatures due to memory limit
    xs_filename = bio_T_fine_path.joinpath('bio_mT_{:03}_xs.npy'.format((100*T).astype(int)))

    def bio_sim_T_vary(key): # fix h=0, N_steps=N**2
        return bio_ric_sim_prime(key, T, zero_field, N**2)
    bio_sim_T_vary_optimized = jax.jit(jax.vmap(bio_sim_T_vary))

    xs, _ = bio_sim_T_vary_optimized(keys_1e5[:10000])
    jnp.save(xs_filename, xs)

100%|██████████| 151/151 [35:07<00:00, 13.96s/it]


Random GRNs: 1000 GRN-replica pairs x finely spaced temperatures

In [5]:
rand_T_fine_path = SAVE_PATH.joinpath('rand_T_fine')
rand_T_fine_path.mkdir(parents=True, exist_ok=True)
rgrns_1e4 = jnp.load(RGRNS_PATH)[:10000]

for T in tqdm(fine_temperatures): # for loop over temperatures due to memory limit
    xs_filename = rand_T_fine_path.joinpath('rand_mT_{:03}_xs.npy'.format((100*T).astype(int)))

    def rand_pair_sim_T_vary(rand_pair): # fix h=0, N_steps=N**2
        return rand_pair_sim_prime(rand_pair, T, zero_field, N**2)
    rand_sim_T_vary_optimized = jax.jit(jax.vmap(rand_pair_sim_T_vary))

    xs, _ = rand_sim_T_vary_optimized((keys_1e5[:10000], rgrns_1e4))
    jnp.save(xs_filename, xs)

100%|██████████| 151/151 [34:06<00:00, 13.56s/it]


## Varying fields
Biological GRN: 1000 replicas x varying fields

In [36]:
bio_h_coarse_path = SAVE_PATH.joinpath('bio_h_vary')
bio_h_coarse_path.mkdir(parents=True, exist_ok=True)

for h in tqdm(varying_fields): # for loop over temperatures due to memory limit
    xs_filename = bio_h_coarse_path.joinpath('bio_h_{}_xs.npy'.format(h.astype(int)))
    vs_filename = bio_h_coarse_path.joinpath('bio_h_{}_vs.npy'.format(h.astype(int)))

    def bio_sim_h_vary(key): # fix T=0, N_steps=N**2
        return bio_ric_sim_prime(key, 0.0, h*nonzero_field, N**2)
    bio_sim_h_vary_optimized = jax.jit(jax.vmap(bio_sim_h_vary))

    xs, vs = bio_sim_h_vary_optimized(keys_1e3)
    jnp.save(xs_filename, xs)
    jnp.save(vs_filename, vs)

100%|██████████| 13/13 [00:17<00:00,  1.31s/it]


## Topological speed limit

Biological GRN: 100000 replicas

In [4]:
tsl_save_path = SAVE_PATH.joinpath('tsl')
tsl_save_path.mkdir(parents=True, exist_ok=True)

bio_many_xs_filename = tsl_save_path.joinpath('bio_many_xs.npy')
bio_many_vs_filename = tsl_save_path.joinpath('bio_many_vs.npy')

def bio_many_sim(key): # fix T=0, h=0, N_steps=N
    return bio_ric_sim_prime(key, 0.0, zero_field, N)
bio_many_optimized = jax.jit(jax.vmap(bio_many_sim))

xs, vs = bio_many_optimized(keys_1e5)
jnp.save(bio_many_xs_filename, xs)
jnp.save(bio_many_vs_filename, vs)

Random GRN: 100000 GRN-replica pairs

In [5]:
rgrns = jnp.load(RGRNS_PATH)

rand_many_xs_filename = tsl_save_path.joinpath('rand_many_xs.npy')
rand_many_vs_filename = tsl_save_path.joinpath('rand_many_vs.npy')

def rand_many_sim(rand_pair): # fix T=0, h=0, N_steps=N
    return rand_pair_sim_prime(rand_pair, 0.0, zero_field, N)
rand_many_optimized = jax.jit(jax.vmap(rand_many_sim))

xs, vs = rand_many_optimized((keys_1e5, rgrns))
jnp.save(rand_many_xs_filename, xs)
jnp.save(rand_many_vs_filename, vs)

In [24]:
# bio_fc_path = SAVE_PATH.joinpath('bio_fc')
# bio_fc_path.mkdir(parents=True, exist_ok=True)

# for T in tqdm(test_temperatures): # for loop over temperatures due to memory limit
#     xs_filename = bio_fc_path.joinpath('bio_mT_{:03}_xs.npy'.format((100*T).astype(int)))

#     def bio_sim_fc(key): # fix h=epsilon, N_steps=N**2
#         return bio_ric_sim_prime_fs(key, T, tiny_field, N**2)
#     bio_sim_fc_optimized = jax.jit(jax.vmap(bio_sim_fc))

#     xs = bio_sim_fc_optimized(keys_1e5[:10000])
#     jnp.save(xs_filename, xs)

100%|██████████| 101/101 [11:11<00:00,  6.65s/it]


In [ ]:
# rand_T_fine_path = SAVE_PATH.joinpath('rand_T_fine')
# rand_T_fine_path.mkdir(parents=True, exist_ok=True)
# rgrns_1e4 = jnp.load(RGRNS_PATH)[:10000]

# for T in tqdm(fine_temperatures): # for loop over temperatures due to memory limit
#     xs_filename = rand_T_fine_path.joinpath('rand_mT_{:03}_xs.npy'.format((100*T).astype(int)))

#     def rand_pair_sim_T_vary(rand_pair): # fix h=0, N_steps=N**2
#         return rand_pair_sim_prime(rand_pair, T, zero_field, N**2)
#     rand_sim_T_vary_optimized = jax.jit(jax.vmap(rand_pair_sim_T_vary))

#     xs, _ = rand_sim_T_vary_optimized((keys_1e5[:10000], rgrns_1e4))
#     jnp.save(xs_filename, xs)